# Notebook 06 — Diagnosing Slow Queries: Your Troubleshooting Playbook
## A Symptom-Based Decision Tree

Your query is slow. Here's how to figure out WHY and WHAT to do about it — without being a DBA.

```
Query is slow
│
├─ High bytes_scanned? ──────────── Partition pruning issue
│   ├─ Filtering by date? ──────── Check natural ordering (Notebook 01)
│   ├─ Filtering by ID/category? ─ Request clustering (Notebook 02)
│   └─ Exact-match lookup? ──────── Request SOS (Notebook 03)
│
├─ Spilling to disk? ────────────── Warehouse too small
│   └─ Use larger warehouse ──────── (Notebook 04)
│
├─ Re-runs just as slow? ────────── Cache not working
│   └─ Replace CURRENT_DATE() ───── Use fixed dates (Notebook 05)
│
└─ Bytes scanned looks fine? ────── Query pattern issue
    ├─ SELECT * ──────────────────── Use specific columns (Notebook 05)
    └─ Joining before filtering ──── Filter first (Notebook 05)
```

In [ ]:
USE ROLE SYSADMIN;
USE WAREHOUSE HOL_XS;
USE SCHEMA JPMC_PERF_HOL_V2.CONSUMER_BANKING;

ALTER SESSION SET USE_CACHED_RESULT = FALSE;

---
## Tool 1: Your Recent Query Performance

Start here — see which of your recent queries scanned the most data or took the longest.

In [ ]:
-- Your slowest queries in the last hour
SELECT
    query_text,
    warehouse_name,
    total_elapsed_time / 1000 AS elapsed_sec,
    bytes_scanned / (1024*1024*1024) AS gb_scanned,
    rows_produced
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY(
    END_TIME_RANGE_START => DATEADD(hour, -1, CURRENT_TIMESTAMP()),
    RESULT_LIMIT => 20
))
WHERE query_type = 'SELECT'
  AND total_elapsed_time > 1000
ORDER BY total_elapsed_time DESC
LIMIT 10;

---
## Tool 2: Diagnose a Specific Slow Query

Pick your slowest query from above. Key diagnostic questions:
- **GB scanned >> rows produced?** → Partition pruning issue
- **Elapsed time high but GB scanned reasonable?** → Spilling or queuing
- **Same query sometimes fast, sometimes slow?** → Cache hit vs miss

In [ ]:
-- Quick health check: scanning efficiency across your recent queries
SELECT
    CASE
        WHEN bytes_scanned / NULLIF(rows_produced, 0) > 1000000 THEN 'EXCESSIVE SCAN (check pruning)'
        WHEN total_elapsed_time > 30000 THEN 'SLOW (check spilling/warehouse)'
        ELSE 'OK'
    END AS diagnosis,
    COUNT(*) AS query_count,
    AVG(bytes_scanned / (1024*1024*1024)) AS avg_gb_scanned,
    AVG(total_elapsed_time / 1000) AS avg_elapsed_sec
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY(
    END_TIME_RANGE_START => DATEADD(hour, -1, CURRENT_TIMESTAMP()),
    RESULT_LIMIT => 50
))
WHERE query_type = 'SELECT'
  AND bytes_scanned > 0
GROUP BY 1
ORDER BY avg_gb_scanned DESC;

---
## Tool 3: Check Warehouse Utilization

Are queries queueing? Is the warehouse right-sized?

In [ ]:
-- Warehouse performance summary
SELECT
    warehouse_name,
    COUNT(*) AS total_queries,
    AVG(total_elapsed_time / 1000) AS avg_elapsed_sec,
    MAX(total_elapsed_time / 1000) AS max_elapsed_sec,
    SUM(bytes_scanned) / (1024*1024*1024*1024) AS total_tb_scanned
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY(
    END_TIME_RANGE_START => DATEADD(hour, -2, CURRENT_TIMESTAMP()),
    RESULT_LIMIT => 100
))
WHERE query_type = 'SELECT'
GROUP BY 1
ORDER BY avg_elapsed_sec DESC;

---
## Quick Reference: What You Learned Today

| Notebook | Problem | Solution | Who Fixes It |
|----------|---------|----------|--------------|
| 01 - Natural Ordering | Date query scans everything | Load data in date order | Data Engineer |
| 02 - Clustering | Non-date filter scans everything | Add cluster key | Data Engineer |
| 03 - Search Optimization | Exact-match lookup is slow | Enable SOS | Data Engineer |
| 04 - Warehouse Sizing | Query spills to disk | Use bigger warehouse | You (or Admin) |
| 05 - Query Tuning | Inefficient SQL patterns | Better SQL habits | **YOU** |

---
## Your Action Items

### Things YOU can fix today:
1. Stop using `SELECT *` — always name your columns
2. Filter before JOINing in complex queries
3. Use fixed dates in recurring dashboards
4. Check Query Profile for spilling → request warehouse resize

### Things to request from your Data Engineering team:
1. "This table is slow on [column] filters" → Ask about clustering
2. "ID lookups take forever" → Ask about Search Optimization
3. "Date queries should be faster" → Ask if data was loaded in order

### How to make the case:
```sql
-- Run this and share with your data engineer:
SELECT query_text, bytes_scanned/(1024*1024*1024) AS gb_scanned, total_elapsed_time/1000 AS sec
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY(
    END_TIME_RANGE_START => DATEADD(hour, -24, CURRENT_TIMESTAMP()), RESULT_LIMIT => 50))
WHERE bytes_scanned > 5000000000 AND query_type = 'SELECT'
ORDER BY bytes_scanned DESC LIMIT 10;
```

This shows them exactly which queries are expensive — data speaks louder than tickets.